# Cod Drumming Classification — Full Pipeline

This notebook walks through every step of classifying cod drumming sounds using **Google AI's Perch 2.0**.

All code uses functions from `src/`.

**Steps:**
1. Load the annotated dataset
2. Explore audio properties
3. Preprocess audio for Perch
4. Extract Perch embeddings
5. Visualise the data
6. Train a classifier
7. Evaluate results

In [ ]:
# Make sure the project root is the working directory so all paths work
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

## 1. Load the Dataset

The annotated data lives in `data/annotated/{split}/{class}/*.wav`.

- **split**: `train`, `val`, or `test`
- **class**: `click`, `other`, `silence`, `vocal`, `water`

`load_dataset` walks this folder structure and returns a list of `AudioSample` objects — each has a file path, a label, and which split it belongs to.

In [ ]:
from src.data.config import DATA_DIR, CLASSES
from src.data.loader import load_dataset

samples = load_dataset(DATA_DIR)
print(f'Total samples loaded: {len(samples)}')

In [ ]:
# Count samples per class per split
from collections import Counter
import pandas as pd

counts = Counter((s.split, s.label) for s in samples)
df = pd.DataFrame(
    [[counts.get((split, cls), 0) for cls in CLASSES] for split in ['train', 'val', 'test']],
    index=['train', 'val', 'test'],
    columns=CLASSES
)
df['TOTAL'] = df.sum(axis=1)
df

## 2. Explore Audio Properties

Before processing, it helps to understand the raw audio:

- **Sample rate**: 96 kHz (hydrophone quality — much higher than normal audio)
- **Duration**: varies widely — clicks are ~0.001 s, silence/other can be several seconds

This matters because Perch expects exactly **5 seconds at 32 kHz**, so every clip must be resampled and padded or trimmed.

In [ ]:
import soundfile as sf

rows = []
for cls in CLASSES:
    cls_samples = [s for s in samples if s.label == cls]
    durations = [sf.info(s.path).duration for s in cls_samples[:10]]
    info = sf.info(cls_samples[0].path)
    rows.append({
        'class': cls,
        'sample_rate_kHz': info.samplerate // 1000,
        'min_duration_s': round(min(durations), 4),
        'max_duration_s': round(max(durations), 4),
        'total_files': len(cls_samples),
    })

pd.DataFrame(rows).set_index('class')

## 3. Preprocess Audio for Perch

Every clip goes through three steps:

1. **Resample** 96 kHz → 32 kHz using a 1:3 integer ratio (fast and lossless with `scipy.signal.resample_poly`)
2. **Center-pad** short clips — zeros added equally on both sides until 5 seconds (160 000 samples)
3. **Center-crop** long clips — take the middle 5 seconds

A click (~0.001 s) becomes a tiny burst of energy centred in silence. This is biologically accurate — the surrounding silence is real.

In [ ]:
from src.data.preprocess import load_and_preprocess
from src.data.config import PERCH_SAMPLE_RATE

for cls in CLASSES:
    s = next(x for x in samples if x.label == cls)
    info = sf.info(s.path)
    processed = load_and_preprocess(s.path)
    print(
        f'{cls:10s}  '
        f'raw: {info.duration:.4f}s @ {info.samplerate//1000}kHz'
        f'  →  processed: {len(processed)/PERCH_SAMPLE_RATE:.1f}s @ {PERCH_SAMPLE_RATE//1000}kHz'
        f'  shape: {processed.shape}'
    )

## 4. Extract Perch Embeddings

**Perch 2.0** is a deep neural network trained by Google on millions of bird sound recordings. Even though it was trained on birds, the acoustic features it learned — spectral patterns, timing, frequency structure — transfer well to other bioacoustic signals, including underwater cod sounds.

For each 5-second clip, Perch outputs a **1280-dimensional embedding**: a compact numerical fingerprint of the sound. This replaces the need to manually engineer features.

Embeddings are saved to `data/embeddings_cache/` after the first run — the model only needs to run once.

In [ ]:
from src.data.config import CACHE_DIR, PERCH_MODEL_NAME
from src.model.perch import load_perch_model

print(f'Loading Perch model: {PERCH_MODEL_NAME} (downloads weights on first run)')
model = load_perch_model(PERCH_MODEL_NAME)

In [ ]:
from src.model.perch import extract_and_cache

embeddings = extract_and_cache(samples, model, CACHE_DIR)

print(f'Embedding shape: {embeddings[str(samples[0].path)].shape}')
print(f'Total embeddings: {len(embeddings)}')

## 5. Visualise the Data

Three views of the data:

- **Waveforms** — raw amplitude over time, one example per class
- **Spectrograms** — frequency content over time (0–5 kHz). This is the standard visualisation in bioacoustics
- **t-SNE** — compresses 1280-dim embeddings to 2D. If classes form separate clusters, the classifier should work well

In [ ]:
from src.visualize.waveforms import plot_waveforms
from IPython.display import Image

plot_waveforms(DATA_DIR, split='train')
Image('models/waveforms.png')

In [ ]:
from src.visualize.spectrograms import plot_spectrograms

plot_spectrograms(DATA_DIR, split='train')
Image('models/spectrograms.png')

In [ ]:
from src.visualize.tsne import plot_tsne

plot_tsne(samples, embeddings)
Image('models/tsne.png')

## 6. Train the Classifier

We train a **logistic regression** on the Perch embeddings.

Why not a neural network?
- With only ~100 training samples, neural networks overfit
- The 1280-dim Perch embeddings are already rich, well-formed features — a simple linear boundary works well
- Logistic regression trains in milliseconds

How it works:
1. Stack all train embeddings into a matrix `X` of shape `(100, 1280)`
2. Encode class labels as integers `y`
3. Fit — the model learns a weighted combination of the 1280 features for each class

We might change to a neural network when number of training samples increase

In [ ]:
from src.training.train import build_arrays, train_classifier, label_names

X_train, y_train = build_arrays(samples, embeddings, 'train')
X_val,   y_val   = build_arrays(samples, embeddings, 'val')
X_test,  y_test  = build_arrays(samples, embeddings, 'test')

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

clf = train_classifier(X_train, y_train, classifier='logistic')
print('Classifier trained.')

## 7. Evaluate Results

We evaluate on both **validation** and **test** sets.

Key metrics:
- **Precision** — of all clips predicted as class X, how many actually were X?
- **Recall** — of all true X clips, how many did we correctly identify?
- **F1** — harmonic mean of precision and recall. The main metric to compare models.

The confusion matrix shows which classes get confused with each other — this points to where more data or better labelling would help.

In [ ]:
from src.training.evaluate import evaluate

evaluate(clf, X_val,  y_val,  label_names(), 'val')
evaluate(clf, X_test, y_test, label_names(), 'test')

In [ ]:
from src.visualize.metrics import plot_class_metrics

plot_class_metrics(clf, X_val, y_val, X_test, y_test, label_names())
Image('models/class_metrics.png')

In [ ]:
Image('models/confusion_test.png')